# U-Net：TGS 盐层二值语义分割

本 notebook 完成数据检查、图像与掩码配对、二值化、训练/验证划分、可视化以及 PyTorch Dataset/DataLoader。
原始数据在 D:\code\Blog\archive\competition_data。保留原始 PNG，在 Dataset 中按需变换，不另存缩放副本。

In [ ]:
from pathlib import Path
from collections import Counter
import sys
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

ROOT = Path(r"D:\code\Blog\archive\competition_data")
TRAIN_IMAGES = ROOT / "train" / "images"
TRAIN_MASKS = ROOT / "train" / "masks"
TEST_IMAGES = ROOT / "test" / "images"
for folder in (TRAIN_IMAGES, TRAIN_MASKS, TEST_IMAGES):
    assert folder.is_dir(), f"缺少目录: {folder}"
print("Python:", sys.executable, "PyTorch:", torch.__version__)

## 1. 配对与审计

训练图片和掩码以同名 PNG 配对。实际掩码混用了 8 位和 16 位格式，前景可能是 65535。
因此统一以非零像素作为盐层，不能仅判断是否等于 255。图像是 RGBA，但颜色通道相同，
本实验转为单通道灰度。

In [ ]:
images = {p.stem: p for p in TRAIN_IMAGES.glob("*.png")}
masks = {p.stem: p for p in TRAIN_MASKS.glob("*.png")}
test_files = sorted(TEST_IMAGES.glob("*.png"))
assert images.keys() == masks.keys(), "图像与掩码文件名不匹配"
ids = sorted(images)
coverage = []
image_modes, mask_modes, values = Counter(), Counter(), set()
for sample_id in ids:
    with Image.open(images[sample_id]) as im, Image.open(masks[sample_id]) as mask:
        assert im.size == mask.size, f"尺寸不匹配: {sample_id}"
        image_modes[im.mode] += 1
        mask_modes[mask.mode] += 1
        a = np.asarray(mask)
        values.update(np.unique(a).tolist())
        coverage.append(np.count_nonzero(a) / a.size)
coverage = np.asarray(coverage, dtype=np.float32)
print("训练配对:", len(ids), "测试图片:", len(test_files))
print("图像模式:", dict(image_modes), "掩码模式:", dict(mask_modes))
print("掩码像素值:", sorted(values))
print("空掩码数:", int(np.count_nonzero(coverage == 0)))
print("覆盖率分位数:", np.quantile(coverage, [0, .25, .5, .75, 1]))
assert values <= {0, 255, 65535}, "发现新像素值，请先检查掩码"

## 2. 查看空掩码、少量盐层和大量盐层

叠加图应与盐层边界对齐。若错位，先检查文件配对与图像变换。

In [ ]:
choices = [int(np.argmin(coverage)),
           int(np.argmin(np.abs(coverage - .10))),
           int(np.argmin(np.abs(coverage - .70)))]
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, idx in enumerate(choices):
    sample_id = ids[idx]
    with Image.open(images[sample_id]) as im, Image.open(masks[sample_id]) as mask:
        x = np.asarray(im.convert("L"))
        y = np.asarray(mask) > 0
    overlay = np.repeat(x[..., None], 3, axis=2).astype(np.float32)
    overlay[y] = overlay[y] * .4 + np.array([153, 0, 0])
    axes[row, 0].imshow(x, cmap="gray", vmin=0, vmax=255)
    axes[row, 1].imshow(y, cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(overlay.astype(np.uint8))
    axes[row, 0].set_ylabel(f"{sample_id}\n覆盖率 {coverage[idx]:.1%}")
    for col, title in enumerate(("原图", "二值掩码", "叠加")):
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 3. 按盐层覆盖率划分训练集与验证集

空掩码单独成组，非空掩码按覆盖率分四组，每组约 80% 训练、20% 验证。
固定随机种子以便复现。先划分，再只对训练集做增强，避免数据泄漏。

In [ ]:
rng = np.random.default_rng(42)
strata = np.where(coverage == 0, 0, np.minimum(np.ceil(coverage * 4).astype(int), 4))
train_ids, val_ids = [], []
for group in sorted(np.unique(strata)):
    group_indices = np.flatnonzero(strata == group)
    rng.shuffle(group_indices)
    n_val = max(1, round(len(group_indices) * .2))
    val_ids.extend(ids[i] for i in group_indices[:n_val])
    train_ids.extend(ids[i] for i in group_indices[n_val:])
rng.shuffle(train_ids)
rng.shuffle(val_ids)
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids) | set(val_ids) == set(ids)
coverage_by_id = dict(zip(ids, coverage))
print("训练:", len(train_ids), "空掩码:", sum(coverage_by_id[i] == 0 for i in train_ids))
print("验证:", len(val_ids), "空掩码:", sum(coverage_by_id[i] == 0 for i in val_ids))

## 4. Dataset 与 DataLoader

U-Net 有三次池化，所以将 101×101 缩放为 128×128。图像用双线性插值。
掩码先统一为 0/255，再用最近邻插值，最后转成 0/1 浮点张量。
模型输入和真值形状都是 [B,1,128,128]。测试集没有掩码。

In [ ]:
class TGSSaltDataset(Dataset):
    def __init__(self, ids, image_dir, mask_dir=None, size=128, augment=False):
        self.ids = list(ids)
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir) if mask_dir is not None else None
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        sample_id = self.ids[index]
        with Image.open(self.image_dir / f"{sample_id}.png") as im:
            im = im.convert("L").resize(
                (self.size, self.size), Image.Resampling.BILINEAR)
            x = np.asarray(im, dtype=np.float32).copy() / 255.0
        if self.mask_dir is None:
            return torch.from_numpy(x[None]).float(), sample_id
        with Image.open(self.mask_dir / f"{sample_id}.png") as mask:
            raw = np.asarray(mask)
            binary = Image.fromarray((raw > 0).astype(np.uint8) * 255, mode="L")
            binary = binary.resize(
                (self.size, self.size), Image.Resampling.NEAREST)
            y = (np.asarray(binary) > 0).astype(np.float32)
        if self.augment and np.random.rand() < .5:
            x = np.ascontiguousarray(x[:, ::-1])
            y = np.ascontiguousarray(y[:, ::-1])
        return (torch.from_numpy(x[None]).float(),
                torch.from_numpy(y[None]).float(), sample_id)

In [ ]:
train_data = TGSSaltDataset(train_ids, TRAIN_IMAGES, TRAIN_MASKS, augment=True)
val_data = TGSSaltDataset(val_ids, TRAIN_IMAGES, TRAIN_MASKS)
test_data = TGSSaltDataset([p.stem for p in test_files], TEST_IMAGES)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=8, shuffle=False, num_workers=0)

batch_x, batch_y, batch_ids = next(iter(train_loader))
print("图像:", tuple(batch_x.shape), batch_x.dtype,
      (batch_x.min().item(), batch_x.max().item()))
print("掩码:", tuple(batch_y.shape), batch_y.dtype,
      torch.unique(batch_y).tolist())
assert batch_x.shape == batch_y.shape == (8, 1, 128, 128)
assert set(torch.unique(batch_y).tolist()) <= {0.0, 1.0}

## 5. 接入 U-Net

本任务是盐层/背景二值分割。U-Net 第一层输入通道应为 1，最后的 1×1 卷积输出通道应为 1。
训练时直接计算 BCEWithLogitsLoss(logits, masks)，不要先手动 sigmoid。
推理时做 sigmoid 后按 0.5 阈值二值化。这里只检验数据与损失函数接口，尚未训练模型。

depths.csv 的深度值可以在后续版本作为额外输入；初版先忽略。
预测测试集时，将 128×128 概率图恢复到原始 101×101，再阈值化。

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()
dummy_logits = torch.zeros_like(batch_y)  # 之后替换为 model(batch_x)
print("期望输出:", tuple(dummy_logits.shape),
      "示例损失:", criterion(dummy_logits, batch_y).item())
print("测试输入:", tuple(test_data[0][0].shape), "ID:", test_data[0][1])